<a href="https://colab.research.google.com/github/warren-research/biodata-inventory-2022/blob/main/full_training_pipeline_with_checkpoints_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 **Full Training Pipeline with Checkpoints (Clean)**

**Complete biomedical ML training pipeline with hybrid checkpointing system**

---

## 📋 **Overview**

This notebook provides a complete training pipeline for the biodata inventory ML models:
- **Classification Model**: Bio-resource paper identification
- **NER Model**: Database name extraction from papers
- **Hybrid Checkpointing**: Local files → Google Drive → Fresh computation
- **Clean Architecture**: Core algorithms with utility functions separated

### **Key Features**
- ✅ **Google Drive First**: Mount drive first for checkpoint access
- ✅ **Unique Session Paths**: No directory cleaning needed - each session isolated
- ✅ **Checkpoint Recovery**: Resume from any major step
- ✅ **GPU Optimization**: Automatic batch size and memory management
- ✅ **Model Archival**: Complete training artifacts preserved
- ✅ **Progress Tracking**: Real-time training progress with todos

### **Execution Order**
1. **Mount Google Drive** - Must be first for checkpoint system
2. **Configure Session** - Unique paths eliminate conflicts
3. **Environment Setup** - Dependencies and utilities
4. **Training Pipeline** - 6-step process with checkpoints
5. **Model Deployment** - Production-ready models
6. **Archive Creation** - Complete artifact preservation

---

## ⚙️ **Configuration & Setup**

In [4]:
# =============================================================================
# STEP 1: MOUNT GOOGLE DRIVE (MUST BE FIRST)
# =============================================================================

# Mount Google Drive for checkpoints and archival
# This MUST be done first to ensure all checkpoint paths work correctly
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("💾 Checkpoint and archive paths are now accessible")
print("🔗 Ready for hybrid checkpointing system")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully
💾 Checkpoint and archive paths are now accessible
🔗 Ready for hybrid checkpointing system


In [ ]:
# =============================================================================
# STEP 2: TRAINING PIPELINE CONFIGURATION
# =============================================================================

import os
import random
import string
from datetime import datetime

# =============================================================================
# CRITICAL CONFIGURATION VARIABLES
# =============================================================================

# Session Management (Generate first for use in paths)
UNIQUE_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

# Directory Management (No cleaning needed with unique paths)
CLEAN_DIRECTORY = False  # Not needed with unique directory names

# Training Configuration
MODEL_NAME = "biomed_roberta_rct500"
HF_MODEL = "allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500"
EPOCHS = 10
BATCH_SIZE = 16  # Will be optimized based on GPU
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0

# Path Configuration
INVENTORY_DIRECTORY = "/content/drive/MyDrive/inventory_2022"
DATA_DIRECTORY = f"{INVENTORY_DIRECTORY}/data"

# Checkpoint Configuration (Note: Google Drive must be mounted first!)
CHECKPOINT_BASE = f"{INVENTORY_DIRECTORY}/training_checkpoints/{UNIQUE_ID}"
USE_CHECKPOINTS = True

# Data Configuration
CLASSIF_DATA = f"{DATA_DIRECTORY}/manual_classifications.csv"
NER_DATA = f"{DATA_DIRECTORY}/manual_ner_extraction.csv"

# Output Directories (All use UNIQUE_ID for session isolation)
CLASSIF_SPLITS_DIR = f"data/classif_splits_full_{UNIQUE_ID}"
NER_SPLITS_DIR = f"data/ner_splits_full_{UNIQUE_ID}"
CLASSIF_OUTPUT_DIR = f"out/classif_train_full_{UNIQUE_ID}"
NER_OUTPUT_DIR = f"out/ner_train_full_{UNIQUE_ID}"
LOG_DIR = f"logs_{UNIQUE_ID}"
BACKUP_DIR = f"model_backups_{UNIQUE_ID}"

# Archive Configuration
ARCHIVE_BASE = f"{INVENTORY_DIRECTORY}/training_archives/{UNIQUE_ID}_full_training"

# =============================================================================
# CONFIGURATION DISPLAY
# =============================================================================

print(f"🔧 Training Session: {UNIQUE_ID}")
print(f"📅 Timestamp: {TIMESTAMP}")
print(f"🤖 Model: {MODEL_NAME} ({HF_MODEL})")
print(f"📊 Training: {EPOCHS} epochs, batch size {BATCH_SIZE}")
print(f"💾 Checkpoints: {CHECKPOINT_BASE}")
print(f"📦 Archive: {ARCHIVE_BASE}")
print(f"🧹 Clean Directory: {CLEAN_DIRECTORY} (not needed with unique paths)")

print(f"\n📁 Path Configuration:")
print(f"   📂 Inventory Directory: {INVENTORY_DIRECTORY}")
print(f"   📂 Data Directory: {DATA_DIRECTORY}")
print(f"   📋 Classification Data: {CLASSIF_DATA}")
print(f"   🏷️ NER Data: {NER_DATA}")

print(f"\n📁 Session-Specific Output Directories:")
print(f"   📋 Classification Splits: {CLASSIF_SPLITS_DIR}")
print(f"   🏷️ NER Splits: {NER_SPLITS_DIR}")
print(f"   📋 Classification Output: {CLASSIF_OUTPUT_DIR}")
print(f"   🏷️ NER Output: {NER_OUTPUT_DIR}")
print(f"   📝 Logs: {LOG_DIR}")
print(f"   💾 Backups: {BACKUP_DIR}")

# Configure training parameters dictionary for utilities
config = {
    'unique_id': UNIQUE_ID,
    'timestamp': TIMESTAMP,
    'model_name': MODEL_NAME,
    'model': HF_MODEL,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'checkpoint_base': CHECKPOINT_BASE,
    'archive_base': ARCHIVE_BASE,
    'inventory_directory': INVENTORY_DIRECTORY,
    'data_directory': DATA_DIRECTORY,
    'optimal_batch_size': BATCH_SIZE  # Will be updated after GPU optimization
}

print("\n" + "="*60)
print("CONFIGURATION COMPLETE - UNIQUE SESSION PATHS")
print("="*60)

In [6]:
# =============================================================================
# DIRECTORY CLEANING SAFETY CHECK (SIMPLIFIED)
# =============================================================================

# With unique directory names, cleaning is not necessary
# Each session gets its own isolated directories

print("✅ Using unique session directories - no cleaning required")
print(f"📁 Session ID: {UNIQUE_ID}")
print("🚀 All output directories are session-specific")
print("🔄 Multiple training sessions can run in parallel")
print("\n✅ Directory management complete - ready to proceed with training")

✅ Using unique session directories - no cleaning required
📁 Session ID: 2025-10-23-pi9dx1
🚀 All output directories are session-specific
🔄 Multiple training sessions can run in parallel

✅ Directory management complete - ready to proceed with training


## 🔧 **Environment Setup**

In [ ]:
# Install required packages with flexible version ranges
!pip install transformers datasets evaluate seqeval scikit-learn pandas numpy nltk

print("✅ Dependencies installed successfully")

# Download NLTK Data
import nltk
import ssl

print("Downloading NLTK data...")

# Handle SSL issues in Colab
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')
print("✅ NLTK data downloaded successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c867dab8263d67e79424c4844519e9a0f5df89d3a50df77b5e29581f71d979e4
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
✅ Dependencies installed successfully


In [ ]:
# Setup working environment
import os
import sys


# Setup Python path
if f'{INVENTORY_DIRECTORY}/' not in sys.path:
    sys.path.append(f'{INVENTORY_DIRECTORY}/')

# Import all utility functions
from src.training_utils import (
    create_directory_structure, check_prerequisites,
    check_local_splits, check_drive_checkpoint,
    load_splits_from_checkpoint, save_splits_to_checkpoint,
    load_training_from_checkpoint, save_training_to_checkpoint,
    verify_config_compatibility, clear_gpu_memory, setup_gpu_optimizations,
    display_split_statistics, display_training_results,
    check_local_deployment, deploy_production_models,
    create_final_archive, show_progress
)

# Verify environment
import torch
import transformers
import datasets
import evaluate

print(f"🐍 Python: {sys.version}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🤗 Transformers: {transformers.__version__}")
print(f"📊 Datasets: {datasets.__version__}")
print(f"📈 Evaluate: {evaluate.__version__}")
print(f"🎯 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

print("\n✅ Environment setup complete")

In [ ]:
# Create directory structure and setup GPU optimizations
create_directory_structure(UNIQUE_ID)

# Setup GPU optimizations and get optimal batch size
scaler, optimal_batch_size = setup_gpu_optimizations()
config['optimal_batch_size'] = optimal_batch_size

print(f"\n🚀 Optimal batch size set to: {optimal_batch_size}")
print("✅ Setup complete - ready for training")

## 🔄 **Training Pipeline**

In [ ]:
# Check prerequisites and initialize progress tracking
check_prerequisites(DATA_DIRECTORY)

# Initialize progress tracking
progress = {
    'data_splits': '⏳',
    'classification_training': '⏳',
    'ner_training': '⏳',
    'evaluation': '⏳',
    'deployment': '⏳',
    'final_archive': '⏳'
}

show_progress(progress, UNIQUE_ID)
print("\n✅ Prerequisites verified - pipeline ready")

In [ ]:
# =============================================================================
# STEP 1: DATA SPLITS GENERATION
# =============================================================================

import json
from pathlib import Path

progress['data_splits'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("📊 Step 1: Data Splits Generation")
print("=" * 40)

# Check for existing local splits
if check_local_splits(CLASSIF_SPLITS_DIR, NER_SPLITS_DIR):
    print("✅ Local data splits found")
    display_split_statistics(CLASSIF_SPLITS_DIR, NER_SPLITS_DIR)

elif USE_CHECKPOINTS and check_drive_checkpoint(f"{CHECKPOINT_BASE}/data_splits", "data_splits"):
    print("📥 Loading data splits from checkpoint...")
    load_splits_from_checkpoint(f"{CHECKPOINT_BASE}/data_splits", CLASSIF_SPLITS_DIR, NER_SPLITS_DIR)
    display_split_statistics(CLASSIF_SPLITS_DIR, NER_SPLITS_DIR)

else:
    print("🔧 Generating fresh data splits...")

    # Generate classification splits
    print("\n📋 Generating classification splits...")
    !python "{INVENTORY_DIRECTORY}/src/class_data_generator.py" \
        -o "{CLASSIF_SPLITS_DIR}" \
        --splits 0.7 0.15 0.15 \
        -r \
        "{CLASSIF_DATA}"

    # Generate NER splits
    print("\n🏷️ Generating NER splits...")
    !python "{INVENTORY_DIRECTORY}/src/ner_data_generator.py" \
        -o "{NER_SPLITS_DIR}" \
        --splits 0.7 0.15 0.15 \
        -r \
        "{NER_DATA}"

    display_split_statistics(CLASSIF_SPLITS_DIR, NER_SPLITS_DIR)

    # Save to checkpoint
    if USE_CHECKPOINTS:
        print("\n💾 Saving splits to checkpoint...")
        save_splits_to_checkpoint(f"{CHECKPOINT_BASE}/data_splits", CLASSIF_SPLITS_DIR, NER_SPLITS_DIR)

        # Save configuration
        Path(f"{CHECKPOINT_BASE}").mkdir(parents=True, exist_ok=True)
        with open(f"{CHECKPOINT_BASE}/config.json", 'w') as f:
            json.dump(config, f, indent=2)
        print("💾 Configuration saved to checkpoint")

progress['data_splits'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ Data splits ready")

In [ ]:
# =============================================================================
# STEP 2: CLASSIFICATION MODEL TRAINING
# =============================================================================

import time

progress['classification_training'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("📋 Step 2: Classification Model Training")
print("=" * 40)

# Check for existing training results
checkpoint_path = f"{CHECKPOINT_BASE}/classification_training"
if Path(f"{CLASSIF_OUTPUT_DIR}/checkpt.pt").exists():
    print("✅ Local classification model found")
    display_training_results(CLASSIF_OUTPUT_DIR, "Classification")

elif USE_CHECKPOINTS and check_drive_checkpoint(checkpoint_path, "classification_training"):
    print("📥 Loading classification training from checkpoint...")

    # Verify configuration compatibility
    if verify_config_compatibility(checkpoint_path, config):
        load_training_from_checkpoint(checkpoint_path, CLASSIF_OUTPUT_DIR)
        display_training_results(CLASSIF_OUTPUT_DIR, "Classification")
    else:
        print("⚠️ Configuration mismatch - proceeding with fresh training")
        checkpoint_path = None

if not Path(f"{CLASSIF_OUTPUT_DIR}/checkpt.pt").exists():
    print("🔧 Starting fresh classification training...")
    print(f"🤖 Model: {HF_MODEL}")
    print(f"📊 Epochs: {EPOCHS}, Batch Size: {optimal_batch_size}, LR: {LEARNING_RATE}")

    start_time = time.time()

    # Run classification training
    !python "{INVENTORY_DIRECTORY}/src/class_train.py" \
        -t "{CLASSIF_SPLITS_DIR}/train_paper_classif.csv" \
        -v "{CLASSIF_SPLITS_DIR}/val_paper_classif.csv" \
        -m "{HF_MODEL}" \
        -ne {EPOCHS} \
        -batch {optimal_batch_size} \
        -rate {LEARNING_RATE} \
        -decay {WEIGHT_DECAY} \
        -o "{CLASSIF_OUTPUT_DIR}" \
        -r

    end_time = time.time()
    duration = (end_time - start_time) / 60
    print(f"\n⏰ Classification training completed in {duration:.1f} minutes")

    # Display results
    display_training_results(CLASSIF_OUTPUT_DIR, "Classification")

    # Save to checkpoint
    if USE_CHECKPOINTS:
        print("\n💾 Saving classification training to checkpoint...")
        save_training_to_checkpoint(checkpoint_path, CLASSIF_OUTPUT_DIR)

# Clear GPU memory
clear_gpu_memory()

progress['classification_training'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ Classification training complete")

In [ ]:
# =============================================================================
# STEP 3: NER MODEL TRAINING
# =============================================================================

progress['ner_training'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("🏷️ Step 3: NER Model Training")
print("=" * 40)

# Check for existing training results
checkpoint_path = f"{CHECKPOINT_BASE}/ner_training"
if Path(f"{NER_OUTPUT_DIR}/checkpt.pt").exists():
    print("✅ Local NER model found")
    display_training_results(NER_OUTPUT_DIR, "NER")

elif USE_CHECKPOINTS and check_drive_checkpoint(checkpoint_path, "ner_training"):
    print("📥 Loading NER training from checkpoint...")

    # Verify configuration compatibility
    if verify_config_compatibility(checkpoint_path, config):
        load_training_from_checkpoint(checkpoint_path, NER_OUTPUT_DIR)
        display_training_results(NER_OUTPUT_DIR, "NER")
    else:
        print("⚠️ Configuration mismatch - proceeding with fresh training")
        checkpoint_path = None

if not Path(f"{NER_OUTPUT_DIR}/checkpt.pt").exists():
    print("🔧 Starting fresh NER training...")
    print(f"🤖 Model: {HF_MODEL}")
    print(f"📊 Epochs: {EPOCHS}, Batch Size: {optimal_batch_size}, LR: {LEARNING_RATE}")

    start_time = time.time()

    # Run NER training
    !python "{INVENTORY_DIRECTORY}/src/ner_train.py" \
        -c f1 \
        -m "{HF_MODEL}" \
        -ne {EPOCHS} \
        -t "{NER_SPLITS_DIR}/train_ner.pkl" \
        -v "{NER_SPLITS_DIR}/val_ner.pkl" \
        -o "{NER_OUTPUT_DIR}" \
        -batch {optimal_batch_size} \
        -rate {LEARNING_RATE} \
        -decay {WEIGHT_DECAY} \
        -r

    end_time = time.time()
    duration = (end_time - start_time) / 60
    print(f"\n⏰ NER training completed in {duration:.1f} minutes")

    # Display results
    display_training_results(NER_OUTPUT_DIR, "NER")

    # Save to checkpoint
    if USE_CHECKPOINTS:
        print("\n💾 Saving NER training to checkpoint...")
        save_training_to_checkpoint(checkpoint_path, NER_OUTPUT_DIR)

# Clear GPU memory
clear_gpu_memory()

progress['ner_training'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ NER training complete")

In [ ]:
# =============================================================================
# STEP 4: MODEL EVALUATION
# =============================================================================

from src.training_utils import display_evaluation_results

progress['evaluation'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("📈 Step 4: Model Evaluation")
print("=" * 40)

# Check for existing evaluation results
checkpoint_path = f"{CHECKPOINT_BASE}/evaluation"
classif_eval_dir = f"{CLASSIF_OUTPUT_DIR}/test_evaluation"
ner_eval_dir = f"{NER_OUTPUT_DIR}/test_evaluation"

if Path(classif_eval_dir).exists() and Path(ner_eval_dir).exists():
    print("✅ Local evaluation results found")

elif USE_CHECKPOINTS and check_drive_checkpoint(checkpoint_path, "evaluation"):
    print("📥 Loading evaluation results from checkpoint...")
    # Load evaluation results (would need custom implementation)

else:
    print("🔧 Running model evaluation...")

    # Evaluate classification model
    print("\n📋 Evaluating classification model...")
    !python "{INVENTORY_DIRECTORY}/src/class_final_eval.py" \
        -o "{classif_eval_dir}" \
        -t "{CLASSIF_SPLITS_DIR}/test_paper_classif.csv" \
        -c "{CLASSIF_OUTPUT_DIR}/checkpt.pt"

    # Evaluate NER model
    print("\n🏷️ Evaluating NER model...")
    !python "{INVENTORY_DIRECTORY}/src/ner_final_eval.py" \
        -o "{ner_eval_dir}" \
        -t "{NER_SPLITS_DIR}/test_ner.pkl" \
        -c "{NER_OUTPUT_DIR}/checkpt.pt"

    if USE_CHECKPOINTS:
        print("\n💾 Saving evaluation results to checkpoint...")
        # Save evaluation results to checkpoint (would need custom implementation)

# Display evaluation results
print("\n📊 Evaluation Results:")
display_evaluation_results(classif_eval_dir, "Classification")
display_evaluation_results(ner_eval_dir, "NER")

progress['evaluation'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ Model evaluation complete")

In [ ]:
# =============================================================================
# STEP 5: MODEL DEPLOYMENT
# =============================================================================

progress['deployment'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("🚀 Step 5: Model Deployment")
print("=" * 40)

# Check if models are already deployed
if check_local_deployment():
    print("✅ Models already deployed to production locations")
else:
    print("🔧 Deploying models to production locations...")
    models_deployed = deploy_production_models(CLASSIF_OUTPUT_DIR, NER_OUTPUT_DIR)
    print(f"\n✅ Deployed {models_deployed}/2 models successfully")

# Verify deployment
if check_local_deployment():
    print("\n📋 Production Model Status:")
    classif_size = Path("out/classif_train_out/article_classifier.pt").stat().st_size / (1024*1024)
    ner_size = Path("out/ner_train_out/named_entity_recognition.pt").stat().st_size / (1024*1024)
    print(f"   🔹 Classification Model: {classif_size:.0f}MB")
    print(f"   🔹 NER Model: {ner_size:.0f}MB")
    print("   🔹 Ready for production inference")
else:
    print("⚠️ Model deployment incomplete")

progress['deployment'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ Model deployment complete")

In [ ]:
# =============================================================================
# STEP 6: FINAL ARCHIVE CREATION
# =============================================================================

progress['final_archive'] = '🔄'
show_progress(progress, UNIQUE_ID)

print("📦 Step 6: Final Archive Creation")
print("=" * 40)

# Check if archive already exists
if Path(ARCHIVE_BASE).exists():
    print(f"✅ Archive already exists: {ARCHIVE_BASE}")
else:
    print("🔧 Creating comprehensive final archive...")
    archived_count = create_final_archive(ARCHIVE_BASE, UNIQUE_ID, config, CLASSIF_OUTPUT_DIR, NER_OUTPUT_DIR)
    print(f"\n✅ Archive created successfully with {archived_count} items")

# Display archive summary
if Path(ARCHIVE_BASE).exists():
    print(f"\n📊 Archive Summary:")
    print(f"   📁 Location: {ARCHIVE_BASE}")
    print(f"   🆔 Session ID: {UNIQUE_ID}")

    # Count archive contents
    archive_files = list(Path(ARCHIVE_BASE).glob('*'))
    total_size = sum(f.stat().st_size for f in archive_files if f.is_file()) / (1024*1024)
    print(f"   📂 Files: {len(archive_files)}")
    print(f"   💾 Size: {total_size:.0f}MB")
    print(f"   📖 Documentation: README.md")

progress['final_archive'] = '✅'
show_progress(progress, UNIQUE_ID)
print("\n✅ Final archive complete")

## 🎉 **Training Complete**

In [ ]:
# =============================================================================
# TRAINING PIPELINE COMPLETION SUMMARY
# =============================================================================

print("🎉" * 20)
print("🎉 TRAINING PIPELINE COMPLETE")
print("🎉" * 20)

# Final progress display
show_progress(progress, UNIQUE_ID)

print(f"\n📋 Training Session Summary:")
print(f"   🆔 Session ID: {UNIQUE_ID}")
print(f"   📅 Timestamp: {TIMESTAMP}")
print(f"   🤖 Model: {MODEL_NAME}")
print(f"   📊 Configuration: {EPOCHS} epochs, batch size {optimal_batch_size}")
print(f"   💾 Checkpoints: {CHECKPOINT_BASE}")
print(f"   📦 Archive: {ARCHIVE_BASE}")

print(f"\n🚀 Production Models Ready:")
print(f"   📋 Classification: out/classif_train_out/article_classifier.pt")
print(f"   🏷️ NER: out/ner_train_out/named_entity_recognition.pt")

print(f"\n📊 Training Artifacts:")
print(f"   📈 Training stats and logs preserved")
print(f"   🔍 Evaluation results available")
print(f"   📖 Complete documentation generated")
print(f"   🔄 Checkpoint system operational")

print(f"\n✅ Training completed successfully!")
print(f"✅ Models ready for production use")
print(f"✅ All artifacts archived with session ID: {UNIQUE_ID}")

print("\n" + "="*50)
print("READY FOR INVENTORY PROCESSING")
print("="*50)
print("\nUse this TRAINING_SESSION_ID in inventory notebooks:")
print(f"TRAINING_SESSION_ID = '{UNIQUE_ID}'")
print("\nThis enables full traceability from training to inventory results.")